# E-Commerce Customer Analytics

## Project Overview

This project examines customer behavior across multiple stages of the e-commerce journey using four independent datasets. The analysis covers user engagement, purchasing behavior, customer value, churn and retention, and post-purchase experience.

The datasets are analyzed separately but provide complementary perspectives on the customer journey, from early platform interaction to purchasing, retention, and customer experience.

The objective is to identify meaningful patterns, evaluate the analytical value of each dataset, and generate insights that can support business decisions related to customer engagement, purchasing behavior, satisfaction, loyalty, and retention.

## Datasets

Four e-commerce datasets were initially evaluated:

1. **User Behavior & Engagement — [E-commerce User Behavior Dataset for AARRR](https://www.kaggle.com/datasets/luyutongsariel/e-commerce-user-behavior-dataset-for-aarrr/data)**  
   Contains customer interaction and transaction features including browsing time, likes, shares, add-to-cart activity, sales, quantity, discounts, profit, and shipping cost.

2. **Sales & Customer Insights — [Sales and Customer Insights Dataset](https://www.kaggle.com/datasets/imranalishahh/sales-and-customer-insights)**  
   Contains customer-value and purchasing metrics such as lifetime value, average order value, purchase frequency, time between purchases, season, purchase timing, and retention strategy.

3. **Customer Churn — [E-commerce Customer Churn Analysis and Prediction](https://www.kaggle.com/datasets/ankitverma2010/ecommerce-customer-churn-analysis-and-prediction)**  
   Focuses on customer retention and churn using variables such as tenure, app usage, days since last order, complaints, satisfaction score, coupons, cashback, and delivery-related factors.

4. **Purchase Experience — [E-Commerce Customer Behavior and Sales Analysis - TR](https://www.kaggle.com/datasets/umuttuygurr/e-commerce-customer-behavior-and-sales-analysis-tr)**  
   Contains transaction and customer-experience features including total amount, discounts, product category, payment method, device type, delivery time, session duration, pages viewed, returning-customer status, and customer rating.

In [1]:
# iomport necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Define data directory
DATA_DIR = Path("../data")

# Load datasets
df_behavior = pd.read_csv(DATA_DIR / "df_behavior.csv",low_memory=False)

df_sales = pd.read_csv(DATA_DIR / "df_sales.csv")

df_churn = pd.read_excel(DATA_DIR / "df_churn.xlsx",sheet_name="E Comm")

df_purchase_experience = pd.read_csv(DATA_DIR / "df_purchase_experience.csv")

# Verify successful loading
datasets = {
    "User Behavior & Engagement": df_behavior,
    "Sales & Customer Insights": df_sales,
    "Customer Churn": df_churn,
    "Purchase Experience": df_purchase_experience,
}

dataset_summary = pd.DataFrame(
    {
        "Dataset": datasets.keys(),
        "Rows": [df.shape[0] for df in datasets.values()],
        "Columns": [df.shape[1] for df in datasets.values()],
    }
)

dataset_summary

,Dataset,Rows,Columns
0,User Behavior & Engagement,51289,27
1,Sales & Customer Insights,10000,15
2,Customer Churn,5630,20
3,Purchase Experience,5000,18


In [3]:
def inspect_df(df, name):
    """Display a compact overview of dataset structure and data quality."""

    print(f"{name}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

    df.info()

    missing = (df.isna().sum().to_frame("Missing").assign(Percent=lambda x: (x["Missing"] / len(df) * 100).round(2)))

    missing = missing[missing["Missing"] > 0].sort_values("Missing", ascending=False)

    print("\nMissing values:")
    display(missing if not missing.empty else "No missing values")

    print(f"\nDuplicate rows: {df.duplicated().sum():,}")

## 1. User Behavior & Engagement

### 1.1 Initial Inspection

The dataset is first reviewed for structure, data types, missing values, duplicate records, and potential data-quality issues.

In [4]:
# show the first 5 rows of the behavior dataset
df_behavior.head().T


,0,1,2,3,4
Customer ID,LS-001,IZ-002,EN-003,AN-004,ON-005
Customer Name,Lane Daniels,Alvarado Kriz,Moon Weien,Sanchez Bergman,Rowe Jackson
Segment,Consumer,Home Office,Consumer,Corporate,Corporate
City,Brisbane,Berlin,Porirua,Kabul,Townsville
State,Queensland,Berlin,Wellington,Kabul,Queensland
Country,Australia,Germany,New Zealand,Afghanistan,Australia
Region,Oceania,Central,Oceania,Central Asia,Oceania
Gender,Male,Male,Male,Male,Male
Age,22,32,21,22,28
Education,Associate Degree,Bachelor,High School,Bachelor,Associate Degree


In [5]:
inspect_df(df_behavior, "User Behavior & Engagement")

User Behavior & Engagement
Shape: 51,289 rows × 27 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51289 entries, 0 to 51288
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Customer ID          51289 non-null  object 
 1   Customer Name        51289 non-null  object 
 2   Segment              51289 non-null  object 
 3   City                 51289 non-null  object 
 4   State                51289 non-null  object 
 5   Country              51289 non-null  object 
 6   Region               51215 non-null  object 
 7   Gender               51289 non-null  object 
 8   Age                  51289 non-null  int64  
 9   Education            51289 non-null  object 
 10  Marital Status       51289 non-null  object 
 11  Order ID             51289 non-null  object 
 12  Order Date           51289 non-null  object 
 13  Months               51289 non-null  object 
 14  Ship Mode            51289

,Missing,Percent
Region,74,0.14
Order Priority,2,0.00
Quantity,1,0.00



Duplicate rows: 0


The dataset contains 51,289 rows and 27 columns. The initial inspection shows that there are no fully duplicated rows. Missing values are limited to three columns: Region, Order Priority, and Quantity. Since the number of missing values is very small compared with the full dataset size, missingness does not appear to be a major issue at this stage.

However, several columns that should be numeric, such as Sales, Quantity, Discount, Profit, and Shipping Cost, are currently stored as object data types. Also, Order Date is stored as an object instead of a datetime variable. These columns need to be inspected and converted before deeper EDA.

### 1.2 Data Type Issues

Several numeric-looking columns were stored as text. I inspected these fields before conversion to identify formatting issues and invalid values.

In [7]:
df_behavior[["Sales", "Quantity", "Discount", "Profit", "Shipping Cost", "Order Date"]].head(10)

,Sales,Quantity,Discount,Profit,Shipping Cost,Order Date
0,$140.00,2,0.05,$46.00,$4.60,2024/9/15
1,$211.00,3,0.03,$112.00,$11.20,2024/6/30
2,$117.00,5,0.01,$31.20,$3.10,2024/5/15
3,$118.00,2,0.05,$26.20,$2.60,2024/9/15
4,$250.00,1,0.04,$160.00,$16.00,2023/9/15
5,$72.00,3,0.04,$24.00,$2.40,2023/2/25
6,$54.00,1,0.05,$54.00,$5.40,2023/9/15
7,$114.00,5,0.02,$22.60,$2.30,2023/3/30
8,$231.00,5,0.03,$116.40,$11.60,2023/9/15
9,$140.00,2,0.02,$54.40,$5.40,2023/4/21


`Sales`, `Profit`, and `Shipping Cost` include currency symbols, while `Quantity` and `Discount` are also stored as text. `Order Date` should be converted to a datetime type.

### 1.3 Cleaning and Converting Numeric Columns

Numeric conversion revealed a small number of invalid values:

- `Sales`: `"0.xf"`
- `Shipping Cost`: `"test"`
- `Quantity`: `"abc"`
- `Discount`: `"xxx"`

Because the issues were limited to individual cells rather than entire records, the invalid values were converted to missing values instead of removing the full rows.

In [6]:
# Convert currency columns to numeric
money_cols = ["Sales", "Profit", "Shipping Cost"]

for col in money_cols:
    df_behavior[col] = (df_behavior[col].str.replace("$", "", regex=False).str.replace(",", "", regex=False))
    df_behavior[col] = pd.to_numeric(df_behavior[col], errors="coerce")

# Convert remaining numeric columns
numeric_cols = ["Quantity", "Discount"]

for col in numeric_cols:
    df_behavior[col] = pd.to_numeric(df_behavior[col], errors="coerce")

In [7]:
df_behavior[["Sales", "Profit", "Shipping Cost", "Quantity", "Discount"]].dtypes

Sales            float64
Profit           float64
Shipping Cost    float64
Quantity         float64
Discount         float64
dtype: object

### 1.4 Converting Order Date

`Order Date` was stored as text and was converted to datetime format to support time-based analysis.

In [8]:
df_behavior["Order Date"] = pd.to_datetime(df_behavior["Order Date"],errors="coerce")

print("Data type:", df_behavior["Order Date"].dtype)
print("Invalid dates:", df_behavior["Order Date"].isna().sum())

Data type: datetime64[ns]
Invalid dates: 0


The Order Date column was successfully converted to datetime format...

### 1.5 Missing Values After Conversion

After correcting data types and converting invalid numeric entries to missing values, I reassessed the amount of missing data.

In [ ]:
missing_behavior = (pd.DataFrame({"Missing Count": df_behavior.isna().sum(),"Missing Percent": (df_behavior.isna().mean() * 100).round(2)})
    .query("`Missing Count` > 0").sort_values("Missing Count", ascending=False))

missing_behavior

,Missing Count,Missing Percent
Region,74,0.14
Quantity,2,0.00
Order Priority,2,0.00
Sales,1,0.00
Discount,1,0.00
Shipping Cost,1,0.00


Missingness is minimal across the dataset. `Region` has the highest missing count with 74 values (about 0.14%), while all other affected columns contain only one or two missing values.

Because the overall missing rate is very small, missing data is not a major data-quality concern for this dataset.

### 1.6 Summary Statistics

After cleaning the data types, I reviewed the main numerical variables to check their ranges and overall distributions.

In [10]:
summary_cols = ["Age","Sales","Quantity","Discount","Profit","Shipping Cost","Browsing Time (min)"]

df_behavior[summary_cols].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,51289.0,31.32,11.90,18.00,22.00,28.00,37.00,70.00
Sales,51288.0,156.43,65.78,33.00,109.00,159.00,218.00,250.00
Quantity,51287.0,3.00,1.41,1.00,2.00,3.00,4.00,5.00
Discount,51288.0,0.03,0.01,0.01,0.02,0.03,0.04,0.05
Profit,51289.0,72.73,49.10,0.50,25.70,67.00,120.60,167.50
Shipping Cost,51288.0,7.27,4.91,0.10,2.60,6.70,12.10,16.80
Browsing Time (min),51289.0,17.28,9.67,1.00,9.30,16.50,24.60,40.00


The numerical variables show plausible ranges after cleaning. Age ranges from 18 to 70, Quantity from 1 to 5, Discount from 0.01 to 0.05, and Browsing Time from 1 to 40 minutes. Sales, Profit, and Shipping Cost also contain positive values within reasonable ranges.

No obvious numerical anomalies are visible from the summary statistics.

### 1.7 Logical Validation

I checked the main numeric and binary variables for impossible or unexpected values after data cleaning.

In [11]:
logical_checks = {
    "Age below 18": (df_behavior["Age"] < 18).sum(),
    "Age above 100": (df_behavior["Age"] > 100).sum(),
    "Sales below 0": (df_behavior["Sales"] < 0).sum(),
    "Quantity less than or equal to 0": (df_behavior["Quantity"] <= 0).sum(),
    "Discount below 0": (df_behavior["Discount"] < 0).sum(),
    "Discount above 1": (df_behavior["Discount"] > 1).sum(),
    "Profit below 0": (df_behavior["Profit"] < 0).sum(),
    "Shipping Cost below 0": (df_behavior["Shipping Cost"] < 0).sum(),
    "Browsing Time below 0": (df_behavior["Browsing Time (min)"] < 0).sum(),
    "Invalid Like values": (~df_behavior["Like"].isin([0, 1])).sum(),
    "Invalid Share values": (~df_behavior["Share"].isin([0, 1])).sum(),
    "Invalid Add to Cart values": (~df_behavior["Add to Cart"].isin([0, 1])).sum()
}

pd.DataFrame.from_dict(logical_checks, orient="index", columns=["Count"])

,Count
Age below 18,0
Age above 100,0
Sales below 0,0
Quantity less than or equal to 0,0
Discount below 0,0
Discount above 1,0
Profit below 0,0
Shipping Cost below 0,0
Browsing Time below 0,0
Invalid Like values,0


No invalid values were found in the tested ranges. Numeric variables remained within plausible bounds, and `Like`, `Share`, and `Add to Cart` contained only valid binary values of 0 and 1.

### 1.8 Categorical Consistency Checks

I reviewed the categorical variables for missing values, unexpected categories, and inconsistent labels.

In [12]:
categorical_cols_behavior = ["Segment","Country","Region","Gender","Education","Marital Status","Months","Ship Mode","Product Category","Order Priority"]

categorical_summary = pd.DataFrame({"Unique Values": [df_behavior[col].nunique(dropna=True) for col in categorical_cols_behavior],
                                    "Missing Values": [df_behavior[col].isna().sum() for col in categorical_cols_behavior]
                                    }, index=categorical_cols_behavior)

categorical_summary

,Unique Values,Missing Values
Segment,3,0
Country,84,0
Region,14,74
Gender,2,0
Education,5,0
Marital Status,2,0
Months,12,0
Ship Mode,5,0
Product Category,4,0
Order Priority,4,2


In [13]:
for col in ["Region", "Ship Mode"]:
    print(f"\n{col}")
    display(df_behavior[col].value_counts(dropna=False).to_frame("Count"))


Region


,Count
Region,
Central,10904
South,6673
North,5458
Oceania,4245
North Asia,3822
West,3443
Southeast Asia,3310
EMEA,3257
Africa,3238



Ship Mode


,Count
Ship Mode,
Standard Class,30775
Second Class,10309
First Class,7503
Same Day,2701
45788,1


Two categorical inconsistencies were identified:

- `Region = "So3th"` appeared 58 times and was identified as a typo for `"South"`.
- `Ship Mode = "45788"` appeared once and is not a valid shipping category.

The remaining categorical variables did not show obvious labeling inconsistencies.

### 1.9 Fixing Categorical Inconsistencies

`Region = "So3th"` was corrected to `"South"`. The invalid `Ship Mode = "45788"` value could not be reliably inferred, so it was converted to a missing value.

In [15]:
df_behavior["Region"] = df_behavior["Region"].replace("So3th", "South")
df_behavior["Ship Mode"] = df_behavior["Ship Mode"].replace("45788", np.nan)

In [16]:
print("Remaining 'So3th' values:",
      (df_behavior["Region"] == "So3th").sum())

print("Remaining invalid Ship Mode values:",
      (df_behavior["Ship Mode"] == "45788").sum())

print("Missing Ship Mode values:",
      df_behavior["Ship Mode"].isna().sum())

Remaining 'So3th' values: 0
Remaining invalid Ship Mode values: 0
Missing Ship Mode values: 1


### 1.10 Final Missing-Value Review

After correcting the categorical inconsistencies, I reviewed the remaining missing values before deciding how to handle them.

In [17]:
missing_behavior = (pd.DataFrame({"Missing Count": df_behavior.isna().sum(),"Missing Percent": (df_behavior.isna().mean() * 100).round(3)})
    .query("`Missing Count` > 0").sort_values("Missing Count", ascending=False))

missing_behavior

,Missing Count,Missing Percent
Region,74,0.144
Order Priority,2,0.004
Quantity,2,0.004
Sales,1,0.002
Ship Mode,1,0.002
Discount,1,0.002
Shipping Cost,1,0.002


Missingness remains very limited. `Region` has the highest missing rate at approximately 0.14%, while the other affected variables contain only one or two missing values.

Because the missing values represent only a very small fraction of the dataset, I chose to remove the affected rows rather than introduce imputed or assumed values.

### 1.11 Removing Rows with Missing Values

Because missing values represent only a very small fraction of the dataset, rows containing missing values were removed.

In [18]:
rows_before = len(df_behavior)

df_behavior = df_behavior.dropna().copy()

rows_after = len(df_behavior)
rows_removed = rows_before - rows_after
percent_removed = rows_removed / rows_before * 100

print(f"Rows before: {rows_before:,}")
print(f"Rows after: {rows_after:,}")
print(f"Rows removed: {rows_removed:,}")
print(f"Percent removed: {percent_removed:.2f}%")

Rows before: 51,289
Rows after: 51,207
Rows removed: 82
Percent removed: 0.16%


Only 82 rows (0.16% of the original dataset) were removed, leaving 51,207 complete records for further analysis.

In [19]:
print("Remaining missing values:", df_behavior.isna().sum().sum())

Remaining missing values: 0


### 1.12 Row-Level Structure and ID Checks

I examined `Customer ID` and `Order ID` to understand the unit of observation and determine whether repeated IDs represented duplicate records.

In [20]:
id_summary = pd.DataFrame({"Metric": ["Total Rows","Unique Customer IDs","Unique Order IDs","Fully Duplicated Rows"],
                           "Count": [len(df_behavior),df_behavior["Customer ID"].nunique(),df_behavior["Order ID"].nunique(),df_behavior.duplicated().sum()]})

id_summary

,Metric,Count
0,Total Rows,51207
1,Unique Customer IDs,794
2,Unique Order IDs,51191
3,Fully Duplicated Rows,0


In [21]:
repeated_orders = df_behavior[df_behavior["Order ID"].duplicated(keep=False)].sort_values("Order ID")

print("Rows with repeated Order IDs:", len(repeated_orders))
print("Repeated Order IDs:", repeated_orders["Order ID"].nunique())

repeated_orders[["Order ID", "Customer ID", "Product", "Sales", "Quantity", "Add to Cart"]].head(10)

Rows with repeated Order IDs: 32
Repeated Order IDs: 16


,Order ID,Customer ID,Product,Sales,Quantity,Add to Cart
2013,AU-2024-2024,ER-00235,Car Seat Covers,114.0,4.0,0
2022,AU-2024-2024,RT-002024,Car Seat Covers,114.0,3.0,1
9518,El-2024-2024,TZ-00107,Mouse,111.0,5.0,1
9527,El-2024-2024,ES-00203,Fans,149.0,4.0,0
32528,FA-2024-12024,SA-001257,Jeans,218.0,5.0,0
32537,FA-2024-12024,ER-001763,T - Shirts,248.0,3.0,1
22528,FA-2024-2024,ON-001145,Shirts,196.0,5.0,1
22537,FA-2024-2024,LL-00102,Fossil Watch,159.0,1.0,1
40663,FA-2024-20240,EZ-00254,Sneakers,62.0,5.0,0
40753,FA-2024-20240,LL-00399,Fossil Watch,159.0,3.0,1


The dataset contains repeated customer IDs, which is expected because the same customer can appear across multiple orders or interactions. Most order IDs are unique, and the small number of repeated order IDs do not correspond to fully duplicated rows.

The dataset is therefore best interpreted at the order or interaction level rather than the customer level.

### 1.13 Class Balance of Engagement Variables

I examined the distribution of the binary engagement variables to identify potential class imbalance.

In [22]:
binary_cols_behavior = ["Like", "Share", "Add to Cart"]

balance_summary = pd.DataFrame({"Class 0 (%)": [(df_behavior[col] == 0).mean() * 100 for col in binary_cols_behavior],
                                "Class 1 (%)": [(df_behavior[col] == 1).mean() * 100 for col in binary_cols_behavior]}, index=binary_cols_behavior).round(2)

balance_summary

,Class 0 (%),Class 1 (%)
Like,43.25,56.75
Share,49.45,50.55
Add to Cart,33.30,66.70


`Like` and `Share` are relatively balanced. `Add to Cart` shows a moderate imbalance, with about two-thirds of records in the positive class.

This imbalance is not extreme, but it should be considered if `Add to Cart` is later used as a prediction target.

### 1.14 Preparing an Encoded Version

For exploratory analysis, the cleaned dataset retains the original categorical labels for interpretability. I also created a separate version with selected categorical variables one-hot encoded for later modeling tasks.

Identifier and high-cardinality fields were not encoded at this stage.

In [23]:
categorical_cols_to_encode = ["Segment","Region","Gender","Education","Marital Status","Ship Mode","Product Category","Order Priority"]

df_behavior_model = pd.get_dummies(df_behavior,columns=categorical_cols_to_encode,drop_first=True)

print("Cleaned dataset shape:", df_behavior.shape)
print("Encoded dataset shape:", df_behavior_model.shape)

Cleaned dataset shape: (51207, 27)
Encoded dataset shape: (51207, 48)


In [24]:
# Save cleaned and encoded datasets
df_behavior.to_csv(DATA_DIR / "df_behavior_eda.csv",index=False)

df_behavior.to_pickle(DATA_DIR / "df_behavior_eda.pkl")

df_behavior_model.to_csv(DATA_DIR / "df_behavior_model.csv",index=False)

df_behavior_model.to_pickle(DATA_DIR / "df_behavior_model.pkl")

### 1.15 Dataset 1 Cleaning Summary

The User Behavior & Engagement dataset was reduced from 51,289 to 51,207 rows after cleaning, with 82 rows removed (0.16%).

Key cleaning steps included:

- converting `Sales`, `Profit`, `Shipping Cost`, `Quantity`, and `Discount` to numeric types;
- converting `Order Date` to datetime;
- converting invalid numeric entries to missing values;
- correcting `Region = "So3th"` to `"South"`;
- converting the invalid `Ship Mode = "45788"` value to missing;
- removing the small number of remaining incomplete rows;
- confirming that no fully duplicated rows remained;
- reviewing repeated customer and order IDs to understand the row-level structure.

Two prepared versions were created:

- `df_behavior_eda`: cleaned data with original categorical labels preserved;
- `df_behavior_model`: a one-hot encoded version for later modeling.

The final cleaned dataset contains 51,207 rows and 27 columns and is ready for exploratory analysis.

## 2. Sales & Customer Insights

### 2.1 Initial Inspection

The dataset was reviewed for structure, data types, missing values, and duplicate records.

In [25]:
df_sales.head().T

,0,1,2,3,4
Customer_ID,CUST_9HOS83,CUST_AJU17N,CUST_11XNYF,CUST_IGH8G3,CUST_OK6PUM
Product_ID,PROD_IK97D1,PROD_UNN7KP,PROD_0XEW2W,PROD_3IIAJN,PROD_VMIWD2
Transaction_ID,TRANS_II1DZG,TRANS_9HJF7I,TRANS_OT96OM,TRANS_45V00G,TRANS_ZAK760
Purchase_Frequency,17,10,3,12,18
Average_Order_Value,172.57,64.89,120.38,70.34,42.39
Most_Frequent_Category,Electronics,Clothing,Sports,Clothing,Electronics
Time_Between_Purchases,45,6,23,5,10
Region,South America,South America,Asia,North America,North America
Churn_Probability,0.98,0.66,0.6,0.78,0.52
Lifetime_Value,952.81,5427.51,3994.8,4880.01,8557.7


In [26]:
inspect_df(df_sales, "Sales & Customer Insights")

Sales & Customer Insights
Shape: 10,000 rows × 15 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer_ID               10000 non-null  object 
 1   Product_ID                10000 non-null  object 
 2   Transaction_ID            10000 non-null  object 
 3   Purchase_Frequency        10000 non-null  int64  
 4   Average_Order_Value       10000 non-null  float64
 5   Most_Frequent_Category    10000 non-null  object 
 6   Time_Between_Purchases    10000 non-null  int64  
 7   Region                    10000 non-null  object 
 8   Churn_Probability         10000 non-null  float64
 9   Lifetime_Value            10000 non-null  float64
 10  Launch_Date               10000 non-null  object 
 11  Peak_Sales_Date           10000 non-null  object 
 12  Season                    10000 non-null  object 
 13  Pre

'No missing values'


Duplicate rows: 0


The dataset contains 10,000 rows and 15 columns, with no missing values or fully duplicated rows. The numeric variables already use appropriate numeric data types.

`Launch_Date` and `Peak_Sales_Date` are stored as text and require conversion to datetime.

### 2.2 Data Type Review

I reviewed the column data types to identify variables that required conversion before further analysis.

In [41]:
df_sales.dtypes

Customer_ID                  object
Product_ID                   object
Transaction_ID               object
Purchase_Frequency            int64
Average_Order_Value         float64
Most_Frequent_Category       object
Time_Between_Purchases        int64
Region                       object
Churn_Probability           float64
Lifetime_Value              float64
Launch_Date                  object
Peak_Sales_Date              object
Season                       object
Preferred_Purchase_Times     object
Retention_Strategy           object
dtype: object

The numeric variables already use appropriate numeric types, while the identifier and categorical fields are correctly stored as text.

The only data type issues are `Launch_Date` and `Peak_Sales_Date`, which are currently stored as text and should be converted to datetime.

### 2.3 Date Conversion

`Launch_Date` and `Peak_Sales_Date` were converted from text to datetime format. Invalid date values were coerced to missing values so the conversion could also serve as a validation check.

In [27]:
date_cols_sales = ["Launch_Date", "Peak_Sales_Date"]

for col in date_cols_sales:
    df_sales[col] = pd.to_datetime(df_sales[col],errors="coerce")

date_check = pd.DataFrame({"Data Type": df_sales[date_cols_sales].dtypes.astype(str),
                           "Missing Values": df_sales[date_cols_sales].isna().sum()})

date_check

,Data Type,Missing Values
Launch_Date,datetime64[ns],0
Peak_Sales_Date,datetime64[ns],0


Both date columns were successfully converted to datetime format without creating any missing values, indicating that no invalid date entries were present.

### 2.4 Summary Statistics

I reviewed the main numerical variables to check their ranges and overall distributions after data type cleaning.

In [28]:
numeric_cols_sales = ["Purchase_Frequency","Average_Order_Value","Time_Between_Purchases","Churn_Probability","Lifetime_Value"]

df_sales[numeric_cols_sales].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Purchase_Frequency,10000.0,9.96,5.53,1.00,5.00,10.00,15.00,19.00
Average_Order_Value,10000.0,110.01,52.10,20.01,65.26,109.93,155.48,199.96
Time_Between_Purchases,10000.0,46.89,24.71,5.00,25.00,47.00,68.00,89.00
Churn_Probability,10000.0,0.50,0.29,0.00,0.25,0.50,0.75,1.00
Lifetime_Value,10000.0,5031.93,2865.42,100.16,2518.78,5012.18,7526.23,9999.76


The numerical variables fall within plausible ranges. `Purchase_Frequency` ranges from 1 to 19, `Average_Order_Value` from 20.01 to 199.96, and `Time_Between_Purchases` from 5 to 89 days.

`Churn_Probability` remains within the expected 0–1 range, while `Lifetime_Value` ranges from 100.16 to 9,999.76. No obvious numerical anomalies are visible from the summary statistics.

### 2.5 Logical Validation

I checked the main numerical variables for impossible or unexpected values after cleaning.

In [29]:
logical_checks_sales = {
    "Purchase_Frequency below 0": (df_sales["Purchase_Frequency"] < 0).sum(),
    "Average_Order_Value below 0": (df_sales["Average_Order_Value"] < 0).sum(),
    "Time_Between_Purchases below 0": (df_sales["Time_Between_Purchases"] < 0).sum(),
    "Churn_Probability below 0": (df_sales["Churn_Probability"] < 0).sum(),
    "Churn_Probability above 1": (df_sales["Churn_Probability"] > 1).sum(),
    "Lifetime_Value below 0": (df_sales["Lifetime_Value"] < 0).sum()}

logical_checks_sales_df = pd.DataFrame.from_dict(logical_checks_sales, orient="index", columns=["Count"])
logical_checks_sales_df

,Count
Purchase_Frequency below 0,0
Average_Order_Value below 0,0
Time_Between_Purchases below 0,0
Churn_Probability below 0,0
Churn_Probability above 1,0
Lifetime_Value below 0,0


No invalid values were found in the tested numerical ranges. All monetary and frequency variables were non-negative, and `Churn_Probability` remained within the expected 0–1 range.

### 2.6 Categorical Consistency Checks

I reviewed the categorical variables for unexpected categories, inconsistent labels, and missing values.

In [30]:
categorical_cols_sales = ["Most_Frequent_Category","Region","Season","Preferred_Purchase_Times","Retention_Strategy"]

for col in categorical_cols_sales:
    print(f"\n{col}")
    display(df_sales[col].value_counts(dropna=False).to_frame("Count"))


Most_Frequent_Category


,Count
Most_Frequent_Category,
Electronics,2567
Clothing,2510
Home,2476
Sports,2447



Region


,Count
Region,
Europe,2547
South America,2527
Asia,2472
North America,2454



Season


,Count
Season,
Winter,2534
Summer,2521
Fall,2502
Spring,2443



Preferred_Purchase_Times


,Count
Preferred_Purchase_Times,
Evening,3433
Afternoon,3354
Morning,3213



Retention_Strategy


,Count
Retention_Strategy,
Loyalty Program,3364
Email Campaign,3359
Discount,3277


The categorical variables contain consistent and expected labels, with no missing values or obvious formatting inconsistencies. No categorical corrections are required.

### 2.7 Row-Level Structure and ID Checks

I examined the identifier columns to determine the unit of observation and check for repeated IDs.

In [31]:
id_summary_sales = pd.DataFrame({"Metric": ["Total Rows","Unique Customer IDs","Unique Product IDs","Unique Transaction IDs","Fully Duplicated Rows"],
                                 "Count": [len(df_sales),df_sales["Customer_ID"].nunique(),df_sales["Product_ID"].nunique(),df_sales["Transaction_ID"].nunique(),df_sales.duplicated().sum()]
})

id_summary_sales

,Metric,Count
0,Total Rows,10000
1,Unique Customer IDs,10000
2,Unique Product IDs,10000
3,Unique Transaction IDs,10000
4,Fully Duplicated Rows,0


All three identifier columns are unique across the 10,000 records, and no fully duplicated rows were found.

This indicates that each row represents a unique customer-product-transaction record in this dataset.

### 2.8 Churn Probability Distribution

`Churn_Probability` is a continuous variable rather than a binary class label. I reviewed its distribution to understand how values are spread across the 0–1 range.

In [33]:
churn_distribution = (df_sales["Churn_Probability"].value_counts(bins=5, sort=False).to_frame("Count"))

churn_distribution

,Count
"(-0.002, 0.2]",2015
"(0.2, 0.4]",2009
"(0.4, 0.6]",2040
"(0.6, 0.8]",1936
"(0.8, 1.0]",2000


`Churn_Probability` is distributed fairly evenly across the 0–1 range, with each interval containing roughly 20% of the records.

This is not a traditional class-balance check because the variable is continuous, but it confirms that churn-risk values are not concentrated in a narrow portion of the range.

### 2.9 Preparing an Encoded Version

The cleaned dataset retains the original categorical labels for exploratory analysis. A separate one-hot encoded version was created for potential modeling tasks.

In [34]:
categorical_cols_to_encode_sales = ["Most_Frequent_Category","Region","Season","Preferred_Purchase_Times","Retention_Strategy"]

df_sales_model = pd.get_dummies(df_sales,columns=categorical_cols_to_encode_sales,drop_first=True)

print("Cleaned dataset shape:", df_sales.shape)
print("Encoded dataset shape:", df_sales_model.shape)

Cleaned dataset shape: (10000, 15)
Encoded dataset shape: (10000, 23)


In [35]:
# Save cleaned and encoded datasets
df_sales.to_csv(DATA_DIR / "df_sales_eda.csv",index=False)

df_sales.to_pickle(DATA_DIR / "df_sales_eda.pkl")

df_sales_model.to_csv(DATA_DIR / "df_sales_model.csv",index=False)

df_sales_model.to_pickle(DATA_DIR / "df_sales_model.pkl")

### 2.10 Dataset 2 Cleaning Summary

The Sales & Customer Insights dataset contains 10,000 rows and 15 original columns, with no missing values or fully duplicated rows.

Key preparation steps included:

- converting `Launch_Date` and `Peak_Sales_Date` to datetime;
- confirming valid numerical ranges, including `Churn_Probability` between 0 and 1;
- verifying that categorical labels were consistent;
- confirming that `Customer_ID`, `Product_ID`, and `Transaction_ID` are unique across all records;
- reviewing the distribution of `Churn_Probability`;
- creating a separate one-hot encoded version for later modeling.

Two prepared versions were created:

- `df_sales_eda`: cleaned data with original categorical labels preserved;
- `df_sales_model`: a one-hot encoded version for potential modeling.

The dataset is technically clean and ready for exploratory analysis.

## 3. Customer Churn

### 3.1 Initial Inspection

The dataset was reviewed for structure, data types, missing values, and duplicate records.

In [36]:
df_churn.head().T

,0,1,2,3,4
CustomerID,50001,50002,50003,50004,50005
Churn,1,1,1,1,1
Tenure,4.0,NaN,NaN,0.0,0.0
PreferredLoginDevice,Mobile Phone,Phone,Phone,Phone,Phone
CityTier,3,1,1,3,1
WarehouseToHome,6.0,8.0,30.0,15.0,12.0
PreferredPaymentMode,Debit Card,UPI,Debit Card,Debit Card,CC
Gender,Female,Male,Male,Male,Male
HourSpendOnApp,3.0,3.0,2.0,2.0,NaN
NumberOfDeviceRegistered,3,4,4,4,3


In [37]:
inspect_df(df_churn, "Customer Churn")

Customer Churn
Shape: 5,630 rows × 20 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   object 
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   object 
 7   Gender                       5630 non-null   object 
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   object 
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                

,Missing,Percent
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46



Duplicate rows: 0


The dataset contains 5,630 rows and 20 columns, with no fully duplicated rows.

Unlike the previous datasets, several numerical variables contain missing values. The largest missing count occurs in `DaySinceLastOrder` with 307 missing values, while six other variables contain approximately 250–265 missing values each.

The target variable `Churn` has no missing values.

### 3.2 Data Type Review

I reviewed the column data types to determine whether any variables required conversion before further analysis.

In [57]:
df_churn.dtypes

CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
dtype: object

The numerical and categorical variables are stored in appropriate data types, so no immediate type conversion is required.

Several count-based variables are stored as `float64` because they contain missing values. These can be reconsidered after missing-value handling.

### 3.3 Missing-Value Analysis

Several numerical variables contain missing values. I examined both the amount of missingness and whether missing values were associated with differences in churn behavior before deciding how to handle them.

In [38]:
missing_summary = (pd.DataFrame({"Missing Count": df_churn.isna().sum(),"Missing Percent": (df_churn.isna().mean() * 100).round(2)})
                   .query("`Missing Count` > 0").sort_values("Missing Count", ascending=False))

missing_summary

,Missing Count,Missing Percent
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46


In [59]:
missing_count_per_row = df_churn.isna().sum(axis=1)
missing_count_per_row.value_counts().sort_index()

0    3774
1    1856
Name: count, dtype: int64

In [39]:
missing_cols = missing_summary.index.tolist()

missing_churn_summary = []

for col in missing_cols:
    missing_churn_summary.append({
        "Column": col,
        "Missing Count": df_churn[col].isna().sum(),
        "Churn Rate When Missing": df_churn.loc[df_churn[col].isna(), "Churn"].mean(),
        "Churn Rate When Not Missing": df_churn.loc[df_churn[col].notna(), "Churn"].mean()
    })

missing_churn_summary = pd.DataFrame(missing_churn_summary)

missing_churn_summary[["Churn Rate When Missing", "Churn Rate When Not Missing"]] = missing_churn_summary[["Churn Rate When Missing", "Churn Rate When Not Missing"]].round(3)

missing_churn_summary

,Column,Missing Count,Churn Rate When Missing,Churn Rate When Not Missing
0,DaySinceLastOrder,307,0.176,0.168
1,OrderAmountHikeFromlastYear,265,0.053,0.174
2,Tenure,264,0.307,0.162
3,OrderCount,258,0.070,0.173
4,CouponUsed,256,0.031,0.175
5,HourSpendOnApp,255,0.227,0.166
6,WarehouseToHome,251,0.335,0.161


Seven numerical variables contain missing values, with missing rates ranging from approximately 4.5% to 5.5%.

Each affected row contains only one missing value. However, missingness is not equally related to churn across variables. For example, customers with missing `Tenure` or `WarehouseToHome` show substantially higher churn rates than customers with observed values.

Because missingness may contain predictive information, these rows were not removed at this stage. Missing-value handling is deferred until the modeling workflow.

### 3.4 Summary Statistics

I reviewed the numerical variables to examine their ranges, central tendencies, and potential unusual values before applying logical validation checks.

In [41]:
numeric_cols_churn = [col for col in df_churn.select_dtypes(include="number").columns if col != "CustomerID"]

df_churn[numeric_cols_churn].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Churn,5630.0,0.17,0.37,0.0,0.00,0.00,0.00,1.00
Tenure,5366.0,10.19,8.56,0.0,2.00,9.00,16.00,61.00
CityTier,5630.0,1.65,0.92,1.0,1.00,1.00,3.00,3.00
WarehouseToHome,5379.0,15.64,8.53,5.0,9.00,14.00,20.00,127.00
HourSpendOnApp,5375.0,2.93,0.72,0.0,2.00,3.00,3.00,5.00
NumberOfDeviceRegistered,5630.0,3.69,1.02,1.0,3.00,4.00,4.00,6.00
SatisfactionScore,5630.0,3.07,1.38,1.0,2.00,3.00,4.00,5.00
NumberOfAddress,5630.0,4.21,2.58,1.0,2.00,3.00,6.00,22.00
Complain,5630.0,0.28,0.45,0.0,0.00,0.00,1.00,1.00
OrderAmountHikeFromlastYear,5365.0,15.71,3.68,11.0,13.00,15.00,18.00,26.00


The numerical variables generally fall within plausible ranges. `Churn` is binary, with an average of approximately 0.17, indicating that about 17% of customers churned.

Some variables, including `WarehouseToHome`, `NumberOfAddress`, and `DaySinceLastOrder`, have maximum values substantially above their medians. These values are not necessarily errors but warrant logical validation and later distribution analysis.

### 3.5 Logical Validation

I checked the numerical variables for values outside their expected ranges, including invalid binary values, negative counts, and scores outside their defined scales.

In [42]:
logical_checks = {
    "Churn not in 0/1": (~df_churn["Churn"].isin([0, 1])).sum(),
    "Complain not in 0/1": (~df_churn["Complain"].isin([0, 1])).sum(),
    "Tenure below 0": (df_churn["Tenure"] < 0).sum(),
    "CityTier outside 1-3": (~df_churn["CityTier"].isin([1, 2, 3])).sum(),
    "WarehouseToHome below 0": (df_churn["WarehouseToHome"] < 0).sum(),
    "HourSpendOnApp below 0": (df_churn["HourSpendOnApp"] < 0).sum(),
    "NumberOfDeviceRegistered below 1": (df_churn["NumberOfDeviceRegistered"] < 1).sum(),
    "SatisfactionScore outside 1-5": (~df_churn["SatisfactionScore"].isin([1, 2, 3, 4, 5])).sum(),
    "NumberOfAddress below 1": (df_churn["NumberOfAddress"] < 1).sum(),
    "OrderAmountHikeFromlastYear below 0": (df_churn["OrderAmountHikeFromlastYear"] < 0).sum(),
    "CouponUsed below 0": (df_churn["CouponUsed"] < 0).sum(),
    "OrderCount below 0": (df_churn["OrderCount"] < 0).sum(),
    "DaySinceLastOrder below 0": (df_churn["DaySinceLastOrder"] < 0).sum(),
    "CashbackAmount below 0": (df_churn["CashbackAmount"] < 0).sum()}
logical_checks_df = pd.DataFrame.from_dict(logical_checks, orient="index",columns=["Invalid Count"])
logical_checks_df

,Invalid Count
Churn not in 0/1,0
Complain not in 0/1,0
Tenure below 0,0
CityTier outside 1-3,0
WarehouseToHome below 0,0
HourSpendOnApp below 0,0
NumberOfDeviceRegistered below 1,0
SatisfactionScore outside 1-5,0
NumberOfAddress below 1,0
OrderAmountHikeFromlastYear below 0,0


No invalid values were found in the tested numerical ranges. The binary variables contain only 0 and 1, the rating and tier variables remain within their expected scales, and no unexpected negative values were identified.

### 3.6 Categorical Consistency Checks

I reviewed the categorical variables for duplicated meanings, inconsistent labels, and unexpected values.

In [43]:
categorical_cols_churn = ["PreferredLoginDevice","PreferredPaymentMode","Gender","PreferedOrderCat","MaritalStatus"]

for col in categorical_cols_churn:
    print(f"\n{col}")
    display(df_churn[col].value_counts(dropna=False).to_frame("Count"))


PreferredLoginDevice


,Count
PreferredLoginDevice,
Mobile Phone,2765
Computer,1634
Phone,1231



PreferredPaymentMode


,Count
PreferredPaymentMode,
Debit Card,2314
Credit Card,1501
E wallet,614
UPI,414
COD,365
CC,273
Cash on Delivery,149



Gender


,Count
Gender,
Male,3384
Female,2246



PreferedOrderCat


,Count
PreferedOrderCat,
Laptop & Accessory,2050
Mobile Phone,1271
Fashion,826
Mobile,809
Grocery,410
Others,264



MaritalStatus


,Count
MaritalStatus,
Married,2986
Single,1796
Divorced,848


Several categories appear to represent the same underlying meaning:

- `Phone` and `Mobile Phone` in `PreferredLoginDevice`;
- `CC` and `Credit Card` in `PreferredPaymentMode`;
- `COD` and `Cash on Delivery` in `PreferredPaymentMode`;
- `Mobile` and `Mobile Phone` in `PreferedOrderCat`.

`Gender` and `MaritalStatus` do not show obvious labeling inconsistencies.

In [44]:
df_churn["PreferredLoginDevice"] = (df_churn["PreferredLoginDevice"].replace({"Phone": "Mobile Phone"}))

df_churn["PreferredPaymentMode"] = (df_churn["PreferredPaymentMode"].replace({"CC": "Credit Card","COD": "Cash on Delivery"}))

df_churn["PreferedOrderCat"] = (df_churn["PreferedOrderCat"].replace({"Mobile": "Mobile Phone"}))

In [45]:
print("Remaining 'Phone' values:",(df_churn["PreferredLoginDevice"] == "Phone").sum())

print("Remaining 'CC' values:",(df_churn["PreferredPaymentMode"] == "CC").sum())

print("Remaining 'COD' values:",(df_churn["PreferredPaymentMode"] == "COD").sum())

print("Remaining 'Mobile' order categories:",(df_churn["PreferedOrderCat"] == "Mobile").sum())

Remaining 'Phone' values: 0
Remaining 'CC' values: 0
Remaining 'COD' values: 0
Remaining 'Mobile' order categories: 0


The duplicated category labels were standardized successfully, reducing equivalent categories to consistent labels before further analysis.

### 3.7 Row-Level Structure and ID Checks

I examined `CustomerID` to determine the unit of observation and check whether customers appear more than once in the dataset.

In [46]:
id_summary_churn = pd.DataFrame({"Metric": ["Total Rows","Unique Customer IDs","Fully Duplicated Rows"],
                                 "Count": [len(df_churn),df_churn["CustomerID"].nunique(),df_churn.duplicated().sum()]})

id_summary_churn

,Metric,Count
0,Total Rows,5630
1,Unique Customer IDs,5630
2,Fully Duplicated Rows,0


All 5,630 customer IDs are unique, and no fully duplicated rows were found.

This confirms that the dataset is structured at the customer level, with one record per customer.

### 3.8 Target Class Balance

I examined the distribution of the `Churn` target to assess class imbalance before modeling.

In [47]:
churn_counts = df_churn["Churn"].value_counts()
churn_percent = (df_churn["Churn"].value_counts(normalize=True).mul(100).round(2))

churn_balance = pd.DataFrame({"Count": churn_counts,"Percent": churn_percent})

churn_balance

,Count,Percent
Churn,,
0,4682,83.16
1,948,16.84


The target is imbalanced: 83.16% of customers did not churn, while 16.84% did.

Because of this imbalance, future model evaluation should consider metrics such as precision, recall, F1-score, and the confusion matrix rather than relying on accuracy alone.

### 3.9 Preparing EDA and Modeling Versions

Missing values were preserved in the EDA version to avoid changing the original distributions.

For the modeling version, missing-value indicator columns were added before median imputation because the earlier analysis showed that missingness may contain information related to churn.

In [48]:
df_churn_eda = df_churn.copy()
df_churn_model = df_churn.copy()

missing_cols_churn = ["DaySinceLastOrder","OrderAmountHikeFromlastYear","Tenure","OrderCount","CouponUsed","HourSpendOnApp","WarehouseToHome"]

for col in missing_cols_churn:
    df_churn_model[f"{col}_was_missing"] = (df_churn_model[col].isna().astype(int))

for col in missing_cols_churn:
    df_churn_model[col] = df_churn_model[col].fillna(df_churn_model[col].median())

print("EDA version shape:", df_churn_eda.shape)
print("Modeling version shape:", df_churn_model.shape)

EDA version shape: (5630, 20)
Modeling version shape: (5630, 27)


### 3.10 Preparing an Encoded Modeling Version

The categorical variables were one-hot encoded only in the modeling dataset. The EDA version retains the original category labels for interpretability.

In [49]:
categorical_cols_churn = ["PreferredLoginDevice","PreferredPaymentMode","Gender","PreferedOrderCat","MaritalStatus"]

df_churn_model = pd.get_dummies(df_churn_model,columns=categorical_cols_churn,drop_first=True)

print("EDA version shape:", df_churn_eda.shape)
print("Modeling version shape:", df_churn_model.shape)

EDA version shape: (5630, 20)
Modeling version shape: (5630, 34)


In [50]:
# Save cleaned and modeling datasets
df_churn_eda.to_csv(DATA_DIR / "df_churn_eda.csv",index=False)

df_churn_eda.to_pickle(DATA_DIR / "df_churn_eda.pkl")

df_churn_model.to_csv(DATA_DIR / "df_churn_model.csv",index=False)

df_churn_model.to_pickle(DATA_DIR / "df_churn_model.pkl")

### 3.11 Dataset 3 Cleaning Summary

The Customer Churn dataset contains 5,630 customer-level records and 20 original columns, with no fully duplicated rows.

Key preparation steps included:

- reviewing missing values across seven numerical variables;
- preserving missing values in the EDA version because missingness showed potential relationships with churn;
- adding missing-value indicators and applying median imputation in the modeling version;
- confirming valid numerical ranges and binary target values;
- standardizing duplicated categorical labels;
- confirming that each `CustomerID` represents a unique customer;
- identifying class imbalance in the `Churn` target, with 16.84% of customers in the positive class;
- one-hot encoding categorical variables for the modeling version.

Two prepared versions were created:

- `df_churn_eda`: retains original categorical labels and missing values for exploratory analysis;
- `df_churn_model`: includes missing-value indicators, median imputation, and one-hot encoded categorical variables for modeling.

The dataset is ready for exploratory churn analysis and later predictive modeling.

## 4. Purchase Experience

### 4.1 Initial Inspection

The dataset was reviewed for structure, data types, missing values, and duplicate records.

In [51]:
df_purchase_experience.head().T

,0,1,2,3,4
Order_ID,ORD_001337,ORD_004885,ORD_004507,ORD_000645,ORD_000690
Customer_ID,CUST_01337,CUST_04885,CUST_04507,CUST_00645,CUST_00690
Date,2023-01-01,2023-01-01,2023-01-01,2023-01-01,2023-01-01
Age,27,42,43,32,40
Gender,Female,Male,Female,Male,Female
City,Bursa,Konya,Ankara,Istanbul,Istanbul
Product_Category,Toys,Toys,Food,Electronics,Sports
Unit_Price,54.28,244.9,48.15,804.06,755.61
Quantity,1,1,5,1,5
Discount_Amount,0.0,0.0,0.0,229.28,0.0


In [52]:
inspect_df(df_purchase_experience, "Purchase Experience")

Purchase Experience
Shape: 5,000 rows × 18 columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Order_ID                  5000 non-null   object 
 1   Customer_ID               5000 non-null   object 
 2   Date                      5000 non-null   object 
 3   Age                       5000 non-null   int64  
 4   Gender                    5000 non-null   object 
 5   City                      5000 non-null   object 
 6   Product_Category          5000 non-null   object 
 7   Unit_Price                5000 non-null   float64
 8   Quantity                  5000 non-null   int64  
 9   Discount_Amount           5000 non-null   float64
 10  Total_Amount              5000 non-null   float64
 11  Payment_Method            5000 non-null   object 
 12  Device_Type               5000 non-null   object 
 13  Session_Dur

'No missing values'


Duplicate rows: 0


The dataset contains 5,000 rows and 18 columns, with no missing values or fully duplicated rows.

Most numerical and categorical variables are already stored in appropriate data types. The main data type issue is `Date`, which is currently stored as text and should be converted to datetime.

`Is_Returning_Customer` is already stored as a boolean variable.

### 4.2 Data Type Review and Date Conversion

I reviewed the column data types and converted `Date` from text to datetime format.

In [76]:
df_purchase_experience.dtypes

Order_ID                     object
Customer_ID                  object
Date                         object
Age                           int64
Gender                       object
City                         object
Product_Category             object
Unit_Price                  float64
Quantity                      int64
Discount_Amount             float64
Total_Amount                float64
Payment_Method               object
Device_Type                  object
Session_Duration_Minutes      int64
Pages_Viewed                  int64
Is_Returning_Customer          bool
Delivery_Time_Days            int64
Customer_Rating               int64
dtype: object

In [53]:
df_purchase_experience["Date"] = pd.to_datetime(df_purchase_experience["Date"],errors="coerce")

print("Data type:", df_purchase_experience["Date"].dtype)
print("Invalid dates:", df_purchase_experience["Date"].isna().sum())

Data type: datetime64[ns]
Invalid dates: 0


The `Date` column was successfully converted to datetime without creating any missing values.

All remaining variables are already stored in appropriate data types for the current analysis.

### 4.3 Summary Statistics

I reviewed the main numerical variables to examine their ranges, central tendencies, and potential unusual values.

In [54]:
numeric_cols_purchase = ["Age","Unit_Price","Quantity","Discount_Amount","Total_Amount","Session_Duration_Minutes","Pages_Viewed","Delivery_Time_Days","Customer_Rating"]

df_purchase_experience[numeric_cols_purchase].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Age,5000.0,35.03,11.08,18.00,27.00,35.00,42.00,75.00
Unit_Price,5000.0,455.83,712.48,5.18,76.59,182.95,513.93,7159.45
Quantity,5000.0,2.22,1.40,1.00,1.00,2.00,3.00,5.00
Discount_Amount,5000.0,24.85,88.39,0.00,0.00,0.00,8.76,1525.55
Total_Amount,5000.0,983.11,1898.98,7.87,122.52,337.91,979.70,22023.90
Session_Duration_Minutes,5000.0,14.57,8.67,1.00,8.00,13.00,19.00,73.00
Pages_Viewed,5000.0,8.98,2.80,1.00,7.00,9.00,11.00,24.00
Delivery_Time_Days,5000.0,6.50,3.46,1.00,4.00,6.00,8.00,25.00
Customer_Rating,5000.0,3.90,1.13,1.00,3.00,4.00,5.00,5.00


Most numerical variables fall within plausible ranges. `Age`, `Quantity`, `Pages_Viewed`, `Delivery_Time_Days`, and `Customer_Rating` show reasonable values.

`Unit_Price`, `Discount_Amount`, and `Total_Amount` have maximum values substantially above their medians, suggesting right-skewed distributions that can be explored further during EDA.

### 4.4 Logical Validation

I checked the numerical variables for impossible or unexpected values based on their expected ranges.

In [55]:
logical_checks_purchase = {
    "Age below 18":(df_purchase_experience["Age"] < 18).sum(),
    "Age above 100":(df_purchase_experience["Age"] > 100).sum(),
    "Unit Price below 0":(df_purchase_experience["Unit_Price"] < 0).sum(),
    "Quantity <= 0":(df_purchase_experience["Quantity"] <= 0).sum(),
    "Discount Amount below 0":(df_purchase_experience["Discount_Amount"] < 0).sum(),
    "Total Amount below 0":(df_purchase_experience["Total_Amount"] < 0).sum(),
    "Session Duration <= 0":(df_purchase_experience["Session_Duration_Minutes"] <= 0).sum(),
    "Pages Viewed <= 0":(df_purchase_experience["Pages_Viewed"] <= 0).sum(),
    "Delivery Time <= 0":(df_purchase_experience["Delivery_Time_Days"] <= 0).sum(),
    "Customer Rating outside 1-5":(~df_purchase_experience["Customer_Rating"].between(1, 5)).sum()
}

pd.DataFrame.from_dict(logical_checks_purchase,orient="index",columns=["Invalid Count"])

,Invalid Count
Age below 18,0
Age above 100,0
Unit Price below 0,0
Quantity <= 0,0
Discount Amount below 0,0
Total Amount below 0,0
Session Duration <= 0,0
Pages Viewed <= 0,0
Delivery Time <= 0,0
Customer Rating outside 1-5,0


No invalid values were found in the tested numerical ranges. Prices and monetary values are non-negative, quantities and activity measures are positive, and customer ratings remain within the expected 1–5 scale.

### 4.5 Categorical Consistency Checks

I reviewed the categorical variables for unexpected categories, inconsistent labels, and missing values.

In [56]:
categorical_cols_purchase = ["Gender","City","Product_Category","Payment_Method","Device_Type","Is_Returning_Customer"]

for col in categorical_cols_purchase:
    print(f"\n{col}")
    display(df_purchase_experience[col].value_counts(dropna=False).to_frame("Count"))


Gender


,Count
Gender,
Female,2492
Male,2435
Other,73



City


,Count
City,
Istanbul,1284
Ankara,735
Izmir,600
Bursa,496
Adana,378
Antalya,374
Gaziantep,349
Konya,317
Kayseri,257



Product_Category


...

,Count
Payment_Method,
Credit Card,2012
Debit Card,1265
Digital Wallet,965
Bank Transfer,510
Cash on Delivery,248



Device_Type


,Count
Device_Type,
Mobile,2795
Desktop,1711
Tablet,494



Is_Returning_Customer


,Count
Is_Returning_Customer,
True,2990
False,2010


The categorical variables contain consistent and expected labels. No obvious spelling inconsistencies, duplicated category meanings, or missing categorical values were identified.

No categorical corrections are required.

### 4.6 Row-Level Structure and ID Checks

I examined `Order_ID` and `Customer_ID` to determine the unit of observation and check for repeated records.

In [57]:
id_summary_purchase = pd.DataFrame({
    "Metric": ["Total Rows","Unique Order IDs","Unique Customer IDs","Fully Duplicated Rows"],
    "Count": [len(df_purchase_experience),df_purchase_experience["Order_ID"].nunique(),df_purchase_experience["Customer_ID"].nunique(),df_purchase_experience.duplicated().sum()]
})
id_summary_purchase

,Metric,Count
0,Total Rows,5000
1,Unique Order IDs,5000
2,Unique Customer IDs,5000
3,Fully Duplicated Rows,0


Both `Order_ID` and `Customer_ID` are unique across all 5,000 records, and no fully duplicated rows were found.

Each row therefore represents a unique customer order and its associated purchase experience.

### 4.7 Preparing an Encoded Version

The cleaned dataset retains the original categorical labels for exploratory analysis. A separate one-hot encoded version was created for later modeling tasks.

Identifier columns were not encoded.

In [58]:
categorical_cols_purchase = ["Gender","City","Product_Category","Payment_Method","Device_Type","Is_Returning_Customer"]

df_purchase_experience_model = pd.get_dummies(df_purchase_experience,columns=categorical_cols_purchase,drop_first=True)

print("Cleaned dataset shape:", df_purchase_experience.shape)
print("Encoded dataset shape:", df_purchase_experience_model.shape)

Cleaned dataset shape: (5000, 18)
Encoded dataset shape: (5000, 37)


In [59]:
# Save cleaned and encoded datasets
df_purchase_experience.to_csv(DATA_DIR / "df_purchase_experience_eda.csv",index=False)

df_purchase_experience.to_pickle(DATA_DIR / "df_purchase_experience_eda.pkl")

df_purchase_experience_model.to_csv(DATA_DIR / "df_purchase_experience_model.csv",index=False)

df_purchase_experience_model.to_pickle(DATA_DIR / "df_purchase_experience_model.pkl")

### 4.8 Dataset 4 Cleaning Summary

The Purchase Experience dataset contains 5,000 rows and 18 original columns, with no missing values or fully duplicated rows.

Key preparation steps included:

- converting `Date` to datetime format;
- confirming valid numerical ranges;
- verifying consistent categorical labels;
- confirming that both `Order_ID` and `Customer_ID` are unique;
- identifying each row as a unique customer order and purchase experience;
- creating a separate one-hot encoded version for later modeling.

Two prepared versions were created:

- `df_purchase_experience_eda`: cleaned data with original categorical labels preserved;
- `df_purchase_experience_model`: a one-hot encoded version for later modeling.

The cleaned dataset is ready for exploratory analysis and predictive modeling.